# Racetrack with Fitted Value Iteration (FVI)

## Warming up
We start with a few useful installs and imports.

FVI is implemented via SB3's DQN: a replay buffer collects transitions, a frozen target network provides Bellman regression targets, and `target_update_interval` controls how often the target snaps to the Q-network (the "fitting" step). The continuous steering action space is discretized via a thin `ActionWrapper`.

In [3]:
# Install environment and agent
!pip install highway-env
!pip install git+https://github.com/DLR-RM/stable-baselines3

import gymnasium as gym
import highway_env
import numpy as np
import os

gym.register_envs(highway_env)

from stable_baselines3 import DQN
from tqdm.notebook import trange

# Discretize the 1-D continuous steering action into N discrete choices.
# Racetrack default: longitudinal=False, lateral=True — action space is Box(-1,1,shape=(1,)).
# Steering values are normalized in [-1, 1], mapping to [-pi/4, pi/4] rad internally.
class DiscreteSteeringWrapper(gym.ActionWrapper):
    def __init__(self, env, n_actions=7):
        super().__init__(env)
        self.steering_values = np.linspace(-1., 1., n_actions, dtype=np.float32)
        self.action_space = gym.spaces.Discrete(n_actions)

    def action(self, act):
        return np.array([self.steering_values[act]])

  Cloning https://github.com/DLR-RM/stable-baselines3 to /tmp/pip-req-build-nksxtat6
  Running command git clone --filter=blob:none --quiet https://github.com/DLR-RM/stable-baselines3 /tmp/pip-req-build-nksxtat6
  Resolved https://github.com/DLR-RM/stable-baselines3 to commit cc20f5af0cfec798d8c8d26bc9886b1a38ead90c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Training
Run tensorboard locally to visualize training.

In [2]:
%tensorboard --logdir "racetrack_fvi"

UsageError: Line magic function `%tensorboard` not found.


In [ ]:
# OccupancyGrid breaks on highway-env 1.10.2 + gymnasium 1.2.3.
# Kinematics gives a flat (5 vehicles x 6 features) observation that works fine with MlpPolicy.
OBS_CONFIG = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["x", "y", "vx", "vy", "cos_h", "sin_h"],
        "normalize": True,
        "absolute": False,
    }
}

def make_env(render_mode=None):
    from highway_env.envs.common.observation import observation_factory
    env = gym.make('racetrack-v0', render_mode=render_mode)
    env.unwrapped.config.update(OBS_CONFIG)
    env.unwrapped.observation_type = observation_factory(
        env.unwrapped, env.unwrapped.config["observation"]
    )
    env.unwrapped.observation_space = env.unwrapped.observation_type.space()
    return DiscreteSteeringWrapper(env)

env = make_env()

model = DQN('MlpPolicy', env,
            policy_kwargs=dict(net_arch=[256, 256]),
            learning_rate=5e-4,
            buffer_size=100000,
            learning_starts=1000,
            batch_size=32,
            gamma=0.8,
            train_freq=1,
            gradient_steps=1,
            target_update_interval=2000,
            verbose=1,
            tensorboard_log='racetrack_fvi/')
model.learn(int(2e6))

In [ ]:
model.save("models/racetrack_fvi_model")

## Testing

Visualize a few episodes.

In [10]:
from gymnasium.wrappers import RecordVideo
from IPython.display import Video, display

MAX_STEPS = 200  # Configurable to prevent infinite loops / make testing easier

env = make_env(render_mode='rgb_array')
env = RecordVideo(env, video_folder='racetrack_fvi_videos', episode_trigger=lambda e: True)

for episode in range(3):
    (obs, info), done, truncated = env.reset(), False, False
    steps = 0
    while not (done or truncated) and steps < MAX_STEPS:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        steps += 1
    print(f"Episode {episode + 1} done ({steps} steps)")
env.close()

for f in sorted(os.listdir('racetrack_fvi_videos')):
    if f.endswith('.mp4'):
        display(Video(os.path.join('racetrack_fvi_videos', f), embed=True))

/home/jahall21/miniconda3/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /mnt/c/Users/hallj/GitHub/CS-238V-Project/cs238v-VSCS/colabs/racetrack_fvi_videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode 1 done (80 steps)
Episode 2 done (200 steps)
Episode 3 done (44 steps)


In [11]:
import numpy as np

N_EVAL = 100
MAX_STEPS = 200
results = []

eval_env = make_env()

for ep in range(N_EVAL):
    obs, info = eval_env.reset()
    done, truncated = False, False
    steps = 0
    while not (done or truncated) and steps < MAX_STEPS:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, truncated, info = eval_env.step(action)
        steps += 1
    results.append({"success": not done, "length": steps})
eval_env.close()

successes = [r["success"] for r in results]
lengths   = [r["length"]  for r in results]

print(f"Success rate : {sum(successes)/N_EVAL*100:.1f}%  ({sum(successes)}/{N_EVAL})")
print(f"Mean length  : {np.mean(lengths):.1f} steps  ({np.mean(lengths)/5:.1f}s)")
print(f"Median length: {np.median(lengths):.1f} steps")

Success rate : 38.0%  (38/100)
Mean length  : 118.3 steps  (23.7s)
Median length: 97.5 steps
